In [16]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Load the parquet file
g_z_all = pd.read_parquet('../historical g spread/bond_z.parquet')

# Now display info - it will show all columns
g_z_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13530 entries, 0 to 13529
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Security_1                 13530 non-null  object  
 1   Security_2                 13530 non-null  object  
 2   Last_Spread                13530 non-null  float64 
 3   Z_Score                    13530 non-null  float64 
 4   Max                        13530 non-null  float64 
 5   Min                        13530 non-null  float64 
 6   Last_vs_Max                13530 non-null  float64 
 7   Last_vs_Min                13530 non-null  float64 
 8   Percentile                 13530 non-null  float64 
 9   XCCY                       13530 non-null  float64 
 10  Custom_Sector_1            13530 non-null  object  
 11  Custom_Sector_2            13530 non-null  object  
 12  Rating_1                   13530 non-null  object  
 13  Rating_2                   1353

In [17]:
# Filter out rows where either column has maturities < 1 year
short_term_buckets = ['0-0.25', '0.25-0.50', '0.50-1']

g_z_all = g_z_all[
    (~g_z_all['Yrs_Mat_Bucket_1'].isin(short_term_buckets)) &
    (~g_z_all['Yrs_Mat_Bucket_2'].isin(short_term_buckets))
]



In [ ]:
# Filter for CAD only
g_z_cad = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD')
]
# Select columns
g_z_select_columns = g_z_all[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_select_columns = g_z_select_columns.sort_values(by='Z_Score', ascending=False)


# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_select_columns.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
157,BCECN 2.9 08/12/26,CM 1.96 04/21/31,20.3,3.3,20.7,-17.5,-0.4,37.8,99.4,Cable/Telco,NVCC Sub Debt,1-2.1,5.1-7.1,"5,000,000","20,000,000",4.0,1.0
158,BMO 2.7 12/09/26,CM 1.96 04/21/31,-8.5,3.3,-6.7,-42.5,-1.8,34.0,99.4,Dep Note,NVCC Sub Debt,1-2.1,5.1-7.1,"2,000,000","20,000,000",5.0,1.0
161,BNS 2.62 12/02/26,CM 1.96 04/21/31,-8.2,3.2,-7.4,-41.2,-0.8,33.0,99.4,Dep Note,NVCC Sub Debt,1-2.1,5.1-7.1,"10,000,000","20,000,000",5.0,1.0
166,ATDBCN 3 5/8 05/13/51,ATDBCN 3.8 01/25/50,15.1,3.1,15.1,-7.7,0.0,22.8,99.6,Non Financial Maples,Non Financial Maples,>25.1,10.1-25.1,nan,nan,nan,nan
167,BCECN 2.9 09/10/29,CM 1.96 04/21/31,49.1,3.1,49.1,20.7,0.0,28.3,99.8,Cable/Telco,NVCC Sub Debt,4.1-5.1,5.1-7.1,"3,000,000","20,000,000",7.0,1.0
174,BCECN 3.6 09/29/27,CM 1.96 04/21/31,35.8,3.1,37.1,3.5,-1.2,32.4,99.4,Cable/Telco,NVCC Sub Debt,2.1-3.1,5.1-7.1,"5,000,000","20,000,000",5.0,1.0
204,BCECN 2.9 09/10/29,BCECN 4 3/4 09/29/44,-86.0,3.0,-86.0,-109.4,0.0,23.4,99.8,Cable/Telco,Cable/Telco,4.1-5.1,10.1-25.1,"3,000,000","5,000,000",7.0,5.0
224,AQNCN 4.09 02/17/27,CM 1.96 04/21/31,32.2,3.0,32.2,-18.0,0.0,50.3,99.8,Power Gen/Renewable,NVCC Sub Debt,1-2.1,5.1-7.1,nan,"20,000,000",nan,1.0
244,BCFERR 4.702 10/23/43,CM 1.96 04/21/31,88.7,2.9,90.2,55.5,-1.6,33.1,99.4,Infastructure,NVCC Sub Debt,10.1-25.1,5.1-7.1,nan,"20,000,000",nan,1.0
247,CCLBCN 3.864 04/13/28,CM 1.96 04/21/31,59.4,2.9,59.6,32.8,-0.1,26.6,99.4,Industrials,NVCC Sub Debt,2.1-3.1,5.1-7.1,"3,000,000","20,000,000",2.0,1.0


In [39]:
# Filter for rows where Custom_Sector_1 equals Custom_Sector_2
g_z_sector = g_z_select_columns[g_z_select_columns['Custom_Sector_1'] == g_z_select_columns['Custom_Sector_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_sector.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
166,ATDBCN 3 5/8 05/13/51,ATDBCN 3.8 01/25/50,15.1,3.1,15.1,-7.7,0.0,22.8,99.6,Non Financial Maples,Non Financial Maples,>25.1,10.1-25.1,nan,nan,nan,nan
204,BCECN 2.9 09/10/29,BCECN 4 3/4 09/29/44,-86.0,3.0,-86.0,-109.4,0.0,23.4,99.8,Cable/Telco,Cable/Telco,4.1-5.1,10.1-25.1,"3,000,000","5,000,000",7.0,5.0
311,BCECN 3.6 09/29/27,BCECN 4 3/4 09/29/44,-99.3,2.7,-99.3,-125.1,0.0,25.8,99.8,Cable/Telco,Cable/Telco,2.1-3.1,10.1-25.1,"5,000,000","5,000,000",5.0,5.0
322,BCECN 2.9 09/10/29,BCECN 4.35 12/18/45,-85.7,2.7,-85.7,-107.7,0.0,22.0,99.8,Cable/Telco,Cable/Telco,4.1-5.1,10.1-25.1,"3,000,000","3,000,000",7.0,5.0
346,BCECN 1.65 08/16/27,BCECN 4 3/4 09/29/44,-132.3,2.6,-132.2,-162.5,-0.2,30.2,99.0,Cable/Telco,Cable/Telco,2.1-3.1,10.1-25.1,nan,"5,000,000",nan,5.0
354,BCECN 3.8 08/21/28,BCECN 4 3/4 09/29/44,-90.5,2.6,-90.5,-108.4,0.0,17.9,99.8,Cable/Telco,Cable/Telco,3.1-4.1,10.1-25.1,"5,000,000","5,000,000",2.0,5.0
384,BCECN 2.9 09/10/29,BCECN 4.45 02/27/47,-84.0,2.5,-84.0,-106.1,0.0,22.1,99.8,Cable/Telco,Cable/Telco,4.1-5.1,10.1-25.1,"3,000,000","5,000,000",7.0,5.0
394,BCECN 3.6 09/29/27,BCECN 4.35 12/18/45,-99.0,2.5,-98.8,-124.0,-0.2,25.0,99.0,Cable/Telco,Cable/Telco,2.1-3.1,10.1-25.1,"5,000,000","3,000,000",5.0,5.0
415,ALACN 4 1/2 08/15/44,ALACN 4.99 10/04/47,9.4,2.4,13.1,-12.6,-3.7,22.0,99.4,Midstream,Midstream,10.1-25.1,10.1-25.1,nan,"3,000,000",nan,nan
419,BCECN 1.65 08/16/27,BCECN 4.35 12/18/45,-132.0,2.4,-131.2,-161.2,-0.8,29.2,98.6,Cable/Telco,Cable/Telco,2.1-3.1,10.1-25.1,nan,"3,000,000",nan,5.0


In [37]:
# Filter for rows where Yrs_Mat_Bucket_1 equals Yrs_Mat_Bucket_2
g_z_term = g_z_select_columns[g_z_select_columns['Yrs_Mat_Bucket_1'] == g_z_select_columns['Yrs_Mat_Bucket_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_term.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
350,BCECN 3 03/17/31,CM 1.96 04/21/31,64.7,2.6,67.2,35.5,-2.5,29.3,99.4,Cable/Telco,NVCC Sub Debt,5.1-7.1,5.1-7.1,"5,000,000","20,000,000",4.0,1.0
398,BCECN 2.9 08/12/26,BCIMCR 3 03/31/27,10.3,2.5,10.3,-7.9,0.0,18.2,99.8,Cable/Telco,Pension,1-2.1,1-2.1,"5,000,000","5,000,000",4.0,5.0
404,ALACN 3.98 10/04/27,CCOCN 2.95 10/21/27,15.4,2.5,15.4,-0.0,0.0,15.4,99.8,Midstream,Other,2.1-3.1,2.1-3.1,"3,000,000","5,000,000",6.0,10.0
415,ALACN 4 1/2 08/15/44,ALACN 4.99 10/04/47,9.4,2.4,13.1,-12.6,-3.7,22.0,99.4,Midstream,Midstream,10.1-25.1,10.1-25.1,nan,"3,000,000",nan,nan
445,BCECN 1.65 08/16/27,BCECN 2.2 05/29/28,-11.2,2.4,-2.6,-34.6,-8.6,23.4,98.6,Cable/Telco,Cable/Telco,2.1-3.1,2.1-3.1,nan,"3,000,000",nan,nan
454,AERMON 3.36 04/24/47,BCECN 4 3/4 09/29/44,-57.0,2.4,-57.0,-83.1,0.0,26.2,99.8,Airport,Cable/Telco,10.1-25.1,10.1-25.1,"3,000,000","5,000,000",5.0,5.0
457,AERMON 3.03 04/21/50,BCECN 4 3/4 09/29/44,-62.5,2.4,-62.1,-85.2,-0.4,22.7,98.6,Airport,Cable/Telco,10.1-25.1,10.1-25.1,"5,000,000","5,000,000",6.0,5.0
490,AERMON 3.918 06/12/45,BCECN 4 3/4 09/29/44,-53.9,2.3,-53.9,-82.6,0.0,28.7,99.8,Airport,Cable/Telco,10.1-25.1,10.1-25.1,"3,000,000","5,000,000",5.0,5.0
516,ATDBCN 3.439 05/13/41,ATDBCN 3.8 01/25/50,17.5,2.3,18.5,-4.8,-0.9,22.3,96.8,Non Financial Maples,Non Financial Maples,10.1-25.1,10.1-25.1,nan,nan,nan,nan
548,ALCTRA 3.458 04/12/49,BCECN 4 3/4 09/29/44,-56.5,2.2,-56.1,-81.5,-0.4,25.1,99.0,Utility,Cable/Telco,10.1-25.1,10.1-25.1,"2,000,000","5,000,000",4.0,5.0
